# GRU Prefetcher V8 -- delta-target redesign

**What changed vs V1-V7** (and why):

| # | Change                                | Source paper / reference |
|---|---------------------------------------|--------------------------|
| 1 | Predict **next delta**, not offset    | Hashemi et al, ICML 2018 |
| 2 | Filter input to **L1 demand misses**  | Liu et al, MEMSYS 2023 (Princeton), Fig 16 |
| 3 | History = 16 (was 4)                  | Princeton MEMSYS 2023 uses 30; we compromise at 16 |
| 4 | Top-K delta vocab (256 classes + OOV) | Hashemi 2018 |
| 5 | PC bucket = 256 (was 4096)            | Cross-binary generalization |
| 6 | Confidence gate at export time        | Voyager 2021 "selective prefetch" |
| 7 | Cosine LR + dropout                   | Standard practice, d2l.ai ch 10 |

**Expected improvement over V4** (which was 1.3% test_off, -8% IPC):
- top-1 delta accuracy: ~15-30% (random baseline = 1/(256+1) = 0.4%)
- trigger rate: ~10-20% (was 58%)
- IPC: closer to baseline or slightly positive

**Optional**: set `WANDB_API_KEY` env var to log to wandb. Otherwise it's silent.

**Inputs needed** (from `projects/legacy_gru_prefetch/scripts/dump_trace.sh`):
- `access_trace.605.mcf_s-994B.csv`  (train)
- `access_trace.620.omnetpp_s-874B.csv` (test, cross-binary)

**Output**:
- `prefetch_list_GRU_V8.txt` -- ready to feed into `projects/legacy_gru_prefetch/scripts/run_nn_replay.sh`
- `gru_v8_summary.json` -- metrics


In [ ]:
# Install optional deps (only wandb is optional; everything else is in Colab)
import subprocess, sys

def pip_install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

pip_install('wandb')


In [ ]:
import os, time, math, json, random, collections
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0); random.seed(0)
print('device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

# Optional wandb
USE_WANDB = False
WANDB_RUN = None
try:
    import wandb
    if os.environ.get('WANDB_API_KEY'):
        USE_WANDB = True
        print('wandb: ENABLED (WANDB_API_KEY found)')
    else:
        print('wandb: disabled (set WANDB_API_KEY env var to enable)')
except ImportError:
    print('wandb: not installed, skipping')


## 1. Configure paths and hyperparameters

Hyperparameters are chosen based on Liu et al., MEMSYS 2023 (Princeton):
their final sweep ended at GRU(hidden=32, embed=32, HIST=30, vocab=128).
We use a slightly larger config (hidden=64, vocab=256) because we want
higher absolute accuracy numbers, at the cost of a 2-3x param count.


In [ ]:
# --- Paths (edit these to point at your dumped CSVs) ---
TRAIN_CSV = 'access_trace.605.mcf_s-994B.csv'
TEST_CSV  = 'access_trace.620.omnetpp_s-874B.csv'

# --- Architecture (Princeton MEMSYS 2023 Fig 16 + slight scale-up) ---
HIST            = 16        # delta history length (Princeton: 30; we use 16)
DELTA_VOCAB_K   = 256       # top-K most frequent deltas (Hashemi 2018 style)
HIDDEN          = 64        # GRU hidden size (Princeton: 32)
NUM_LAYERS      = 1         # 1-layer GRU (Princeton finding)
EMB_DELTA       = 32        # delta embedding dim (Princeton: 32)
EMB_PC          = 16        # PC embedding dim
NUM_PC_BUCKETS  = 256       # bucket PCs (was 4096 in V1-V4)
DROPOUT         = 0.1

# --- Training ---
EPOCHS          = 10
BATCH           = 1024
LR              = 1e-3
LR_MIN          = 1e-5
WEIGHT_DECAY    = 1e-5
GRAD_CLIP       = 1.0

# --- Data ---
USE_L1DM_ONLY   = True      # Princeton finding: L1 demand miss stream beats Full
LINE_BITS       = 6
PAGE_BITS       = 12

# --- Inference / prefetch export ---
CONF_THRESHOLD  = 0.25      # softmax max prob threshold for emitting prefetch
                            # set to 0.0 to emit on every access

VOCAB_SIZE = DELTA_VOCAB_K + 1  # +1 for OOV token (id=0)

# --- Bookkeeping ---
RUN_NAME = f'GRU_V8_h{HIDDEN}_hist{HIST}_vocab{DELTA_VOCAB_K}_conf{CONF_THRESHOLD}'
print(f'config: {RUN_NAME}')
print(f'  train: {TRAIN_CSV}')
print(f'  test : {TEST_CSV}')
print(f'  vocab: {VOCAB_SIZE} (top-{DELTA_VOCAB_K} deltas + 1 OOV)')
print(f'  L1DM only: {USE_L1DM_ONLY}')

if USE_WANDB:
    WANDB_RUN = wandb.init(project='gru-prefetcher-v8', name=RUN_NAME,
        config={'hist': HIST, 'vocab': VOCAB_SIZE, 'hidden': HIDDEN,
                'num_layers': NUM_LAYERS, 'emb_delta': EMB_DELTA,
                'emb_pc': EMB_PC, 'num_pc_buckets': NUM_PC_BUCKETS,
                'epochs': EPOCHS, 'batch': BATCH, 'lr': LR,
                'l1dm_only': USE_L1DM_ONLY, 'conf_threshold': CONF_THRESHOLD,
                'train_csv': TRAIN_CSV, 'test_csv': TEST_CSV})


## 2. Load CSV + (optionally) filter to L1 demand misses

The dumper records every memory access with a `hit` column. Princeton MEMSYS 2023
showed that training only on L1 demand misses (hit=0) gives better IPC than
training on the full access stream. Intuition: L1 hits don't need prefetching,
so including them just dilutes the training signal.

We preserve the **original** index from the CSV so the replayer can still match
prefetches to the right access counter, even after filtering.


In [ ]:
def load_csv(path, use_l1dm_only):
    t0 = time.time()
    df = pd.read_csv(path)
    df['addr'] = df['addr_hex'].apply(
        lambda s: int(s, 16) if isinstance(s, str) and s.startswith('0x') else int(s)
    ).astype('int64')
    df['pc'] = df['pc_hex'].apply(
        lambda s: int(s, 16) if isinstance(s, str) and s.startswith('0x') else int(s)
    ).astype('int64')
    df['hit'] = df['hit'].astype('int8')
    # Preserve the dumper's idx column verbatim -- the replayer keys off this.
    if 'idx' in df.columns:
        df['idx_orig'] = df['idx'].astype('int64')
    else:
        df['idx_orig'] = np.arange(len(df), dtype='int64')

    n_total = len(df)
    if use_l1dm_only:
        df = df[df['hit'] == 0].reset_index(drop=True)
    n_kept = len(df)
    print(f'[load {path}] {n_total:,} -> {n_kept:,} rows ({n_kept/max(1,n_total)*100:.1f}%)'
          f'  uniq PC={df.pc.nunique()}  uniq page={(df.addr//(1<<PAGE_BITS)).nunique()}'
          f'  ({time.time()-t0:.1f}s)')
    return df

df_train = load_csv(TRAIN_CSV, USE_L1DM_ONLY)
df_test  = load_csv(TEST_CSV, USE_L1DM_ONLY)


## 3. Build delta vocabulary from the training trace (Hashemi 2018 style)

Compute per-PC deltas across the entire train CSV, count them, and keep the
top-K (= 256) most frequent. All other deltas get token id 0 = OOV.

This is exactly the trick from Hashemi 2018: the 2^64 address space is impossible
to learn as classification, but the *set of actually-observed deltas* per program
is small. Coverage of top-256 deltas is typically 60-85% on SPEC workloads.


In [ ]:
def build_delta_vocab(df, K):
    addrs = df['addr'].values
    pcs = df['pc'].values
    last = {}
    deltas = []
    for i in range(len(df)):
        pc, a = int(pcs[i]), int(addrs[i])
        if pc in last:
            d = (a - last[pc]) >> LINE_BITS   # delta in cache-line units
            deltas.append(d)
        last[pc] = a
    cnt = collections.Counter(deltas)
    top = [d for d, _ in cnt.most_common(K)]
    delta_to_id = {d: i+1 for i, d in enumerate(top)}     # 1..K (0 = OOV)
    id_to_delta = {i+1: d for i, d in enumerate(top)}
    coverage = sum(cnt[d] for d in top) / max(1, len(deltas))
    print(f'[vocab] built from {len(deltas):,} deltas, '
          f'top-{K} coverage = {coverage*100:.1f}%')
    print(f'[vocab] top-10 deltas (cache lines): {top[:10]}')
    return delta_to_id, id_to_delta, coverage

delta_to_id, id_to_delta, vocab_coverage = build_delta_vocab(df_train, DELTA_VOCAB_K)
if USE_WANDB:
    wandb.log({'vocab/coverage': vocab_coverage})


## 4. Featurize train and test

For each access:
- **input**: per-PC delta history (last 16 deltas, mapped to vocab ids, oldest->newest)
              + bucketed PC
- **label**: the next delta (as vocab id; OOV if not in top-K)

We do NOT shuffle across the time axis -- the train/val split is time-ordered
(first 70% / last 30% of the train CSV).


In [ ]:
def featurize(df, delta_to_id, hist_len):
    addrs = df['addr'].values
    pcs   = df['pc'].values
    idxs  = df['idx_orig'].values
    N = len(df)

    # Per-PC running history of *delta token ids* (so we don't recompute each step)
    last_addrs = {}   # pc -> most recent addr
    hist_ids   = {}   # pc -> list of recent delta ids (oldest->newest)

    X_delta = np.zeros((N, hist_len), dtype=np.int64)
    X_pc    = np.zeros(N, dtype=np.int64)
    Y       = np.zeros(N, dtype=np.int64)
    Idx     = np.zeros(N, dtype=np.int64)
    Cur     = np.zeros(N, dtype=np.int64)
    keep    = np.zeros(N, dtype=bool)

    OOV = 0
    for i in range(N - 1):
        pc, a, a_next = int(pcs[i]), int(addrs[i]), int(addrs[i+1])

        # Update per-PC history: append the delta from the previous access of this PC.
        if pc in last_addrs:
            d = (a - last_addrs[pc]) >> LINE_BITS
            tok = delta_to_id.get(d, OOV)
            h = hist_ids.setdefault(pc, [])
            h.append(tok)
            if len(h) > hist_len:
                hist_ids[pc] = h[-hist_len:]
        last_addrs[pc] = a

        h = hist_ids.get(pc, [])
        if len(h) >= 1:
            # Left-pad with OOV (0) so the most recent token is at position [hist_len-1]
            padded = [OOV] * (hist_len - len(h)) + h[-hist_len:]
            X_delta[i] = padded
            X_pc[i]    = pc % NUM_PC_BUCKETS

            # Label: token id of the upcoming delta
            d_next = (a_next - a) >> LINE_BITS
            Y[i]   = delta_to_id.get(d_next, OOV)
            Idx[i] = idxs[i]
            Cur[i] = a
            keep[i] = True

    return (X_delta[keep], X_pc[keep], Y[keep], Idx[keep], Cur[keep])

print('[featurize] train...')
trX_d, trX_pc, trY, trIdx, trCur = featurize(df_train, delta_to_id, HIST)
print(f'  -> {trX_d.shape[0]:,} examples')

print('[featurize] test ...')
teX_d, teX_pc, teY, teIdx, teCur = featurize(df_test,  delta_to_id, HIST)
print(f'  -> {teX_d.shape[0]:,} examples')

# Quick stat: what fraction of test labels are OOV (= unpredictable for us)?
oov_frac_train = (trY == 0).mean()
oov_frac_test  = (teY == 0).mean()
print(f'[stat] OOV fraction in train labels: {oov_frac_train*100:.1f}%')
print(f'[stat] OOV fraction in test  labels: {oov_frac_test*100:.1f}%')

# Time-split train (70/30)
split = int(0.7 * len(trY))
print(f'[split] train[0:{split:,}]  val[{split:,}:{len(trY):,}]  test_all[{len(teY):,}]')


## 5. PyTorch Dataset + DataLoader

In [ ]:
class PFDS(Dataset):
    def __init__(self, X_d, X_pc, Y):
        self.X_d  = torch.from_numpy(X_d)
        self.X_pc = torch.from_numpy(X_pc)
        self.Y    = torch.from_numpy(Y)
    def __len__(self): return len(self.Y)
    def __getitem__(self, i): return self.X_d[i], self.X_pc[i], self.Y[i]

tr_ds = PFDS(trX_d[:split],  trX_pc[:split],  trY[:split])
va_ds = PFDS(trX_d[split:],  trX_pc[split:],  trY[split:])
te_ds = PFDS(teX_d, teX_pc, teY)

tr_ld = DataLoader(tr_ds, batch_size=BATCH, shuffle=True,  drop_last=True,  num_workers=0)
va_ld = DataLoader(va_ds, batch_size=BATCH, shuffle=False, drop_last=False, num_workers=0)
te_ld = DataLoader(te_ds, batch_size=BATCH, shuffle=False, drop_last=False, num_workers=0)


## 6. GRU model

Architecture follows d2l.ai chapter 10.2 (https://d2l.ai/chapter_recurrent-modern/gru.html)
for the recurrent core. Head is a single linear layer over the delta vocabulary,
conditioned on the GRU's final hidden state concatenated with the PC embedding.

Key sizing decisions:
- **hidden=64**: Princeton MEMSYS 2023 used 32 and got equal-IPC-to-Voyager
  with 6.7x fewer params. We use 64 to have more headroom for the harder
  cross-binary task.
- **single-layer GRU**: same as Princeton's final sweep config.
- **vocab=257**: 256 top deltas + 1 OOV, vs the previous notebook's 64-class
  offset prediction (which is fundamentally chance-level on pointer-chase).


In [ ]:
class GRUPrefetcherV8(nn.Module):
    def __init__(self, vocab_size, num_pc_buckets, hidden, emb_d, emb_pc,
                 num_layers, dropout):
        super().__init__()
        self.embed_delta = nn.Embedding(vocab_size, emb_d, padding_idx=0)
        self.embed_pc    = nn.Embedding(num_pc_buckets, emb_pc)
        self.gru = nn.GRU(emb_d, hidden, num_layers=num_layers,
                          batch_first=True,
                          dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden + emb_pc, vocab_size)
    def forward(self, delta_hist, pc):
        d_emb = self.embed_delta(delta_hist)         # [B, HIST, EMB_D]
        _, h = self.gru(d_emb)                       # h: [num_layers, B, HIDDEN]
        h_last = h[-1]                               # [B, HIDDEN]
        p_emb = self.embed_pc(pc)                    # [B, EMB_PC]
        z = self.dropout(torch.cat([h_last, p_emb], dim=1))
        return self.head(z)                          # [B, vocab_size]

model = GRUPrefetcherV8(VOCAB_SIZE, NUM_PC_BUCKETS, HIDDEN, EMB_DELTA, EMB_PC,
                       NUM_LAYERS, DROPOUT).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'[model] {n_params:,} trainable params')
if USE_WANDB:
    wandb.log({'model/params': n_params})
    wandb.watch(model, log='gradients', log_freq=500)


## 7. Train with cosine LR schedule

Cross-entropy loss. We **don't** weight by 1/freq because we *want* the
model to predict the common deltas correctly -- those are the ones that
contribute most to IPC.

We also report top-5 accuracy: in practice prefetchers often issue the
top-K candidates, so top-K accuracy is the more honest metric.


In [ ]:
def evaluate(model, loader, device, max_k=5):
    model.eval()
    n = 0; top1 = 0; topk = 0
    loss_sum = 0.0
    with torch.no_grad():
        for X_d, X_pc, Y in loader:
            X_d, X_pc, Y = X_d.to(device), X_pc.to(device), Y.to(device)
            logits = model(X_d, X_pc)
            loss = F.cross_entropy(logits, Y, reduction='sum')
            loss_sum += loss.item()
            pred1 = logits.argmax(dim=1)
            top1 += (pred1 == Y).sum().item()
            topk += (logits.topk(max_k, dim=1).indices == Y.unsqueeze(1)).any(dim=1).sum().item()
            n += Y.numel()
    return {'loss': loss_sum/max(1,n),
            'top1': top1/max(1,n),
            f'top{max_k}': topk/max(1,n),
            'n': n}

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
steps_per_epoch = len(tr_ld)
total_steps = EPOCHS * steps_per_epoch
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_steps, eta_min=LR_MIN)

print(f'[train] {EPOCHS} epochs x {steps_per_epoch:,} steps = {total_steps:,} steps')
best_val_top1 = -1.0
t_start = time.time()
history = []

for ep in range(EPOCHS):
    model.train()
    running = 0.0; nb = 0; ok = 0; tot = 0
    ep_start = time.time()
    for X_d, X_pc, Y in tr_ld:
        X_d, X_pc, Y = X_d.to(DEVICE), X_pc.to(DEVICE), Y.to(DEVICE)
        logits = model(X_d, X_pc)
        loss = F.cross_entropy(logits, Y)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step(); sched.step()
        running += loss.item(); nb += 1
        ok += (logits.argmax(1) == Y).sum().item()
        tot += Y.numel()
    train_loss = running / max(1, nb)
    train_acc = ok / max(1, tot)
    val_m = evaluate(model, va_ld, DEVICE)
    elapsed = time.time() - ep_start
    history.append({'epoch': ep+1, 'train_loss': train_loss, 'train_acc': train_acc,
                    **{f'val_{k}': v for k, v in val_m.items()},
                    'lr': sched.get_last_lr()[0], 'epoch_s': elapsed})
    print(f'  ep {ep+1}/{EPOCHS}  loss={train_loss:.3f}  '
          f'train_top1={train_acc:.3f}  val_top1={val_m["top1"]:.3f}  '
          f'val_top5={val_m["top5"]:.3f}  lr={sched.get_last_lr()[0]:.2e}  '
          f'({elapsed:.1f}s)')
    if USE_WANDB:
        wandb.log({'epoch': ep+1, 'train/loss': train_loss,
                   'train/top1': train_acc,
                   'val/top1': val_m['top1'], 'val/top5': val_m['top5'],
                   'val/loss': val_m['loss'], 'lr': sched.get_last_lr()[0]})
    if val_m['top1'] > best_val_top1:
        best_val_top1 = val_m['top1']

print(f'\n[train] total time: {time.time()-t_start:.1f}s')
print(f'[train] best val top-1: {best_val_top1:.4f}')

# Final test evaluation (cross-binary)
test_m = evaluate(model, te_ld, DEVICE)
print(f'\n[test, cross-binary {TEST_CSV}]')
print(f'  top-1: {test_m["top1"]:.4f}  (chance = {1.0/VOCAB_SIZE:.4f})')
print(f'  top-5: {test_m["top5"]:.4f}')
print(f'  N    : {test_m["n"]:,}')
if USE_WANDB:
    wandb.log({'test/top1': test_m['top1'], 'test/top5': test_m['top5']})


## 8. CPU inference latency micro-benchmark

This is the same metric the existing notebooks track, for comparability.
The actual hardware deployment latency is much lower (table lookup, ~1 cycle,
following the DART 2024 / Net2Tab 2026 tabularization path) -- this CPU number
just shows the relative ordering across architectures.


In [ ]:
def bench_cpu_us(m, hist_len, n_iter=2000):
    m_cpu = m.to('cpu').eval()
    d = torch.zeros((1, hist_len), dtype=torch.long)
    pc = torch.zeros((1,), dtype=torch.long)
    with torch.no_grad():
        for _ in range(20): m_cpu(d, pc)
        t = time.perf_counter()
        for _ in range(n_iter): m_cpu(d, pc)
        us = (time.perf_counter() - t) / n_iter * 1e6
    m.to(DEVICE)
    return us

inf_us = bench_cpu_us(model, HIST)
print(f'[bench] CPU inference: {inf_us:.1f} us / call')
if USE_WANDB:
    wandb.log({'bench/cpu_us': inf_us})


## 9. Export prefetch list with confidence gating

For each test access, run the model, take the top-1 predicted delta. If the
softmax max probability is **above** `CONF_THRESHOLD`, decode the delta back to a
byte offset and emit `<addr + delta>` as the prefetch target.

This is the "selective prefetch" trick from Voyager 2021: don't issue prefetches
the model is unsure about. The cost is missed coverage; the benefit is much
less cache pollution.

The output format is `idx 0xhex` per line, exactly what `list_replayer` consumes.


In [ ]:
out_path = f'prefetch_list_GRU_V8.txt'
n_total = 0; n_emit = 0
confs_top1 = []   # for histogram
gate_counts = {'kept': 0, 'dropped_conf': 0, 'dropped_oov': 0}

BS = 4096
model.to(DEVICE).eval()
with open(out_path, 'w') as fh, torch.no_grad():
    for i in range(0, len(teY), BS):
        X_d  = torch.from_numpy(teX_d[i:i+BS]).to(DEVICE)
        X_pc = torch.from_numpy(teX_pc[i:i+BS]).to(DEVICE)
        logits = model(X_d, X_pc)
        probs = F.softmax(logits, dim=1)
        conf, pred_id = probs.max(dim=1)
        pred_id = pred_id.cpu().numpy()
        conf    = conf.cpu().numpy()
        confs_top1.append(conf)
        idxs   = teIdx[i:i+BS]
        curs   = teCur[i:i+BS]
        for j in range(len(pred_id)):
            n_total += 1
            tok = int(pred_id[j])
            c   = float(conf[j])
            # Drop if model predicts OOV (it's saying "I don't know")
            if tok == 0:
                gate_counts['dropped_oov'] += 1
                continue
            # Drop if low confidence
            if c < CONF_THRESHOLD:
                gate_counts['dropped_conf'] += 1
                continue
            d_lines = id_to_delta[tok]
            pf_addr = (int(curs[j]) + (int(d_lines) << LINE_BITS)) & 0xffffffffffffffff
            if pf_addr <= 0: continue
            fh.write(f'{int(idxs[j])} 0x{pf_addr:x}\n')
            n_emit += 1
            gate_counts['kept'] += 1

confs_top1 = np.concatenate(confs_top1) if confs_top1 else np.array([])
trigger_rate = n_emit / max(1, n_total)
print(f'[export] wrote {out_path}')
print(f'  total predictions : {n_total:,}')
print(f'  emitted prefetches: {n_emit:,}  (trigger rate {trigger_rate*100:.1f}%)')
print(f'  dropped (low conf): {gate_counts["dropped_conf"]:,}')
print(f'  dropped (OOV pred): {gate_counts["dropped_oov"]:,}')
print(f'\n[confidence] top-1 softmax max-prob percentiles:')
if len(confs_top1) > 0:
    for p in [10, 25, 50, 75, 90, 95, 99]:
        print(f'  p{p:>2d} = {np.percentile(confs_top1, p):.3f}')
    for thr in [0.10, 0.25, 0.50, 0.75]:
        kept = (confs_top1 >= thr).mean() * 100
        print(f'  >= {thr:.2f}: {kept:.1f}% kept')
if USE_WANDB:
    wandb.log({'export/trigger_rate': trigger_rate,
               'export/n_emitted': n_emit,
               'export/n_total': n_total,
               'export/conf_threshold': CONF_THRESHOLD})


## 10. Write JSON summary

In [ ]:
summary = {
    'run_name': RUN_NAME,
    'config': {
        'hist': HIST, 'vocab_size': VOCAB_SIZE,
        'hidden': HIDDEN, 'num_layers': NUM_LAYERS,
        'emb_delta': EMB_DELTA, 'emb_pc': EMB_PC,
        'num_pc_buckets': NUM_PC_BUCKETS,
        'epochs': EPOCHS, 'batch': BATCH, 'lr': LR,
        'l1dm_only': USE_L1DM_ONLY,
        'conf_threshold': CONF_THRESHOLD,
    },
    'data': {
        'train_csv': TRAIN_CSV,
        'test_csv':  TEST_CSV,
        'vocab_coverage_train': float(vocab_coverage),
        'oov_train_labels': float(oov_frac_train),
        'oov_test_labels':  float(oov_frac_test),
        'n_train': int(split),
        'n_val':   int(len(trY) - split),
        'n_test':  int(len(teY)),
    },
    'metrics': {
        'best_val_top1': float(best_val_top1),
        'test_top1': float(test_m['top1']),
        'test_top5': float(test_m['top5']),
        'chance_baseline': float(1.0 / VOCAB_SIZE),
        'cpu_inf_us': float(inf_us),
    },
    'prefetch_list': {
        'path': out_path,
        'n_emitted': int(n_emit),
        'n_total':   int(n_total),
        'trigger_rate': float(trigger_rate),
    },
    'history': history,
    'params': int(n_params),
}

with open('gru_v8_summary.json', 'w') as fh:
    json.dump(summary, fh, indent=2)
print('saved gru_v8_summary.json')
print()
print('===== HEADLINE =====')
print(f'  test top-1 acc: {test_m["top1"]:.4f}  (chance {1/VOCAB_SIZE:.4f}, '
      f'so {test_m["top1"]/(1/VOCAB_SIZE):.1f}x above chance)')
print(f'  test top-5 acc: {test_m["top5"]:.4f}')
print(f'  trigger rate  : {trigger_rate*100:.1f}%  (target 10-20%)')
print(f'  CPU inference : {inf_us:.0f} us / call')
print()
print('Next steps:')
print('  1. download prefetch_list_GRU_V8.txt to lab machine')
print(f'  2. cd lab; TRACE=620.omnetpp_s-874B \\')
print(f'         PFETCH=$WORKDIR/prefetch_list_GRU_V8.txt \\')
print(f'         MODEL_TAG=GRU_V8 bash projects/legacy_gru_prefetch/scripts/run_nn_replay.sh')

if USE_WANDB:
    wandb.finish()


## 11. (Optional) Ablation -- run with conf_threshold = 0 to see un-gated IPC

This re-exports the prefetch list **without** the confidence gate so you can
measure the IPC delta directly attributable to gating. Keep both files and run
both through ChampSim.

If V8 (gated) IPC > V8_ungated IPC, the gate is helping (less pollution).
If they're equal, the gate isn't doing anything useful (predictions are
uniformly confident or uniformly unconfident).


In [ ]:
ABLATE = True   # set False to skip
if ABLATE:
    out_path_ungated = 'prefetch_list_GRU_V8_ungated.txt'
    n_total_u, n_emit_u = 0, 0
    model.to(DEVICE).eval()
    with open(out_path_ungated, 'w') as fh, torch.no_grad():
        for i in range(0, len(teY), BS):
            X_d  = torch.from_numpy(teX_d[i:i+BS]).to(DEVICE)
            X_pc = torch.from_numpy(teX_pc[i:i+BS]).to(DEVICE)
            logits = model(X_d, X_pc)
            pred_id = logits.argmax(dim=1).cpu().numpy()
            idxs = teIdx[i:i+BS]; curs = teCur[i:i+BS]
            for j in range(len(pred_id)):
                n_total_u += 1
                tok = int(pred_id[j])
                if tok == 0: continue
                d_lines = id_to_delta[tok]
                pf_addr = (int(curs[j]) + (int(d_lines) << LINE_BITS)) & 0xffffffffffffffff
                if pf_addr <= 0: continue
                fh.write(f'{int(idxs[j])} 0x{pf_addr:x}\n')
                n_emit_u += 1
    print(f'[ablate] wrote {out_path_ungated}  '
          f'{n_emit_u:,}/{n_total_u:,} = {n_emit_u/max(1,n_total_u)*100:.1f}% trigger')
